# Run ChurnLab Straight From GitHub — No Zip Uploads

This notebook replaces the old *"zip `src/`, upload it, unzip it"* workflow.

Instead of uploading the code, it:

1. pulls the latest code from **https://github.com/nikhilwankhedee/churn** (`main` branch),
2. installs any missing dependencies from the repo's `requirements.txt`,
3. adds the cloned repo to `sys.path`, and
4. runs the full pipeline (`run_pipeline`) on your chosen dataset.

**You still attach the *data* datasets as Kaggle inputs** (exactly as before, under `/kaggle/input/datasets`) — only the *code* now comes live from GitHub, so it is always up to date.

> Just hit **Run All**.

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/nikhilwankhedee/churn.git"
REPO_BRANCH = "main"
UPDATE_REPO = True            # git pull the latest code on every run

# This notebook automatically runs BOTH modes for EVERY main-pipeline dataset:
#   Phase 1: original (no SMOTE) — 5 datasets × 5 models = 25 experiments
#   Phase 2: smote              — 5 datasets × 5 models = 25 experiments
# The credit_card/telco ABLATION-only study is in run_ablation_study.ipynb —
# this notebook does NOT re-run their main models (they are already valid).
# Set QUICK_DATASET to a dataset name to trial-run just that one first.

QUICK_DATASET = None          # None     -> full sweep (5 datasets, both modes)
                              # e.g. "olist" -> trial-run only olist (both modes)

CHURN_WINDOW_OVERRIDE = None  # None -> dataset default (e.g. 180 days)
SENSITIVITY = False           # additionally run churn-window sensitivity analysis

# ── Environment / paths ──────────────────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if ON_KAGGLE else ("/content" if os.path.isdir("/content") else os.getcwd())
REPO_DIR = os.path.join(WORK_DIR, "churn")

print(f"Environment  : {'KAGGLE' if ON_KAGGLE else ('COLAB' if '/content' in WORK_DIR else 'LOCAL')}")
print(f"Code will go : {REPO_DIR}")


In [ ]:
# ── Clone / update the code from GitHub ───────────────────────────────────────
import io
import os
import sys
import shutil
import subprocess
import zipfile
import urllib.request


def git(cmd, cwd=None):
    print("$ git " + " ".join(cmd))
    subprocess.check_call(["git"] + cmd, cwd=cwd)


def sync_repo(repo_dir, url, branch, update=True):
    """Clone the repo, or pull the latest code if it already exists.
    Falls back to downloading GitHub's zipball when git is unavailable."""
    repo_dir = os.path.abspath(repo_dir)
    parent = os.path.dirname(repo_dir)

    if os.path.isdir(os.path.join(repo_dir, ".git")):
        if not update:
            print(f"→ Using existing repo at {repo_dir} (UPDATE_REPO=False)")
        else:
            try:
                git(["fetch", "origin"], repo_dir)
                git(["checkout", "--quiet", branch], repo_dir)
                git(["pull", "--rebase", "origin", branch], repo_dir)
                print("→ Repo updated to latest.")
            except Exception as exc:
                # network hiccup — keep the local checkout, it still works offline
                print(f"→ Update failed ({exc}); reusing existing checkout.")
        return repo_dir

    try:
        git(["clone", "--branch", branch, url, repo_dir])
        print(f"→ Cloned {url} → {repo_dir}")
    except Exception:
        # git missing or blocked (e.g. no git binary) — download the zipball instead
        print("git clone failed — downloading GitHub zipball instead…")
        if os.path.isdir(repo_dir):
            shutil.rmtree(repo_dir)  # leftover from a partial clone
        os.makedirs(parent, exist_ok=True)
        zip_url = url.replace(".git", "") + f"/archive/refs/heads/{branch}.zip"
        with urllib.request.urlopen(zip_url, timeout=120) as resp:
            data = resp.read()
        extract_dir = os.path.join(parent, "__churn_extract__")
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            zf.extractall(extract_dir)
        extracted = [os.path.join(extract_dir, d) for d in os.listdir(extract_dir)
                     if os.path.isdir(os.path.join(extract_dir, d))]
        shutil.move(extracted[0], repo_dir)
        shutil.rmtree(extract_dir)
        print(f"→ Downloaded {zip_url} → {repo_dir}")
    return repo_dir


REPO_DIR = sync_repo(REPO_DIR, REPO_URL, REPO_BRANCH, update=UPDATE_REPO)

In [ ]:
# ── Install any missing dependencies (from the repo's requirements.txt) ──────
from importlib.util import find_spec

MODULES = {
    "pandas": "pandas", "numpy": "numpy", "scikit-learn": "sklearn",
    "xgboost": "xgboost", "lightgbm": "lightgbm", "imbalanced-learn": "imblearn",
    "shap": "shap", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "scipy": "scipy", "pingouin": "pingouin", "statsmodels": "statsmodels",
    "joblib": "joblib", "tqdm": "tqdm", "pyyaml": "yaml", "typer": "typer",
    "rich": "rich", "openpyxl": "openpyxl",
}
SKIP = {"jupyter", "ipykernel"}  # already provided by the notebook host

missing = []
with open(os.path.join(REPO_DIR, "requirements.txt")) as fh:
    for line in fh:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name = line.split(">=")[0].split("==")[0].split("[")[0].strip()
        if name.lower() in SKIP:
            continue
        mod = MODULES.get(name.lower(), name.lower().replace("-", "_"))
        if find_spec(mod) is None:
            missing.append(name)

if missing:
    print(f"Installing {len(missing)} missing package(s): {missing}")
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        # PEP 668 'externally managed environment' (e.g. Debian/Ubuntu system
        # python) — retry with the OS-sanctioned override flag
        print("pip refused — retrying with --break-system-packages")
        subprocess.check_call(cmd + ["--break-system-packages"])
else:
    print("All pipeline dependencies are already installed.")

In [ ]:
# ── Add the cloned code to the path and verify the framework imports ─────────
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

from src.config import ON_KAGGLE, PROJECT_ROOT, RESULTS_DIR, FIGURES_DIR, MODELS_DIR, PROCESSED_DIR
from src.datasets import list_datasets

print(f"On Kaggle    : {ON_KAGGLE}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Reg datasets : {', '.join(list_datasets())}")

# version of the code we are running (last commit)
try:
    rev = subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1",
                                   "--format=%h  %cs  %s"], text=True).strip()
    print(f"Code version : {rev}")
except Exception:
    pass

In [ ]:
# ── RUN THE FULL RERUN SWEEP: all five main-pipeline datasets ────────────
import time
import pandas as pd
from IPython.display import display
from src.pipeline import run_pipeline

# Main-pipeline rerun scope: user-disjoint fixes (olist, retailrocket) plus the
# clean datasets re-included in the rerun (instacart, lastfm, rees46).
# The credit_card/telco ablation study moved to run_ablation_study.ipynb.
MAIN_TARGETS = ["olist", "retailrocket", "instacart", "lastfm", "rees46"]

# ── Pre-run validation gate (Part 9): abort if anything FAILs ───────────
from src.preflight import run_preflight, gate_all_passed
report = run_preflight(MAIN_TARGETS)
print(report.to_string(index=False))
if not gate_all_passed(report):
    raise RuntimeError("PRE-RUN VALIDATION FAILED — fix inputs/validity before the sweep.")

targets = [QUICK_DATASET] if QUICK_DATASET is not None else MAIN_TARGETS
print(f"Main targets : {', '.join(targets)}  ({len(targets)} datasets x 5 models)")
print(f"window       : {CHURN_WINDOW_OVERRIDE or 'default'} | sensitivity={SENSITIVITY}")
print(f"Phases       : original (no SMOTE) -> smote  ->  {len(targets) * 2 * 5} experiments total")


def _run_one(name, smote):
    return run_pipeline(
        dataset=name,
        sensitivity=SENSITIVITY,
        churn_window_override=CHURN_WINDOW_OVERRIDE,
        use_smote=smote,
    )


sweep_start = time.time()
phase_tables = []
for smote in (False, True):
    mode = "smote" if smote else "original"
    print("=" * 74)
    print(f"PHASE: {mode.upper()}  ({len(targets)} datasets x 5 models = {len(targets) * 5} experiments)")
    print("=" * 74)
    rows = []
    for name in targets:
        t0 = time.time()
        try:
            result = _run_one(name, smote)
            secs = round(result.get("duration_seconds", 0) or (time.time() - t0), 1)
            rows.append({
                "phase": mode, "dataset": name, "status": "OK",
                "best_model": result.get("best_model"),
                "churn_rate": round(result["churn_rate"], 4) if result.get("churn_rate") is not None else None,
                "imbalance": round(result["imbalance_ratio"], 2) if result.get("imbalance_ratio") is not None else None,
                "seconds": secs,
            })
            print(f"  [{mode:8s}] {name:16s} OK    best={result.get('best_model')}  {secs:7.1f}s")
        except Exception as exc:
            msg = str(exc)
            secs = round(time.time() - t0, 1)
            rows.append({"phase": mode, "dataset": name, "status": "FAILED",
                         "best_model": None, "churn_rate": None,
                         "imbalance": None, "seconds": secs})
            print(f"  [{mode:8s}] {name:16s} FAILED: {msg[:120]}")
            if any(k in msg for k in ("No such file", "FileNotFound", "does not exist", "could not be found")):
                print(f"    Raw data not attached for '{name}' - add its Kaggle input under /kaggle/input/datasets.")
    summary = pd.DataFrame(rows)
    phase_tables.append(summary)
    n_ok = int((summary["status"] == "OK").sum())
    display(summary)
    print(f"\n{n_ok}/{len(targets)} dataset(s) OK for {mode.upper()}\n")

full = pd.concat(phase_tables, ignore_index=True)
print("FULL SWEEP SUMMARY (both phases):")
display(full)
total_ok = int((full["status"] == "OK").sum())
elapsed = time.time() - sweep_start
print(f"\n{total_ok}/{full.shape[0]} (dataset x mode) runs completed in {elapsed/60:.1f} min.")
print("Download /kaggle/working - everything is already split per dataset and mode.")


## What you just ran / what to do next

**50 experiments total** — 5 datasets (olist, retailrocket, instacart, lastfm, rees46) x 5 models,
twice (`results/<dataset>/original/` then `results/<dataset>/smote/`).

The **credit_card / telco ablation-only** study is a separate notebook:
`run_ablation_study.ipynb` (their valid main models are left untouched).

Folders produced under `PROJECT_ROOT` (i.e. `/kaggle/working`):

| Folder | Layout |
|---|---|
| `results/` | `results/<dataset>/<mode>/` — model metrics, stat tests, risk, data quality, experiments |
| `figures/` | `figures/<dataset>/<mode>/` — ROC/PR curves, SHAP, segmentation, calibration, behaviour |
| `models/` | `models/<dataset>/<mode>/` — saved fitted models |
| `processed_data/` | `processed_data/<dataset>/<mode>/` — cleaned, leakage-free folds |
| `results/preflight/` | `preflight_report_<timestamp>.csv` — the pre-run validation gate |

`<mode>` is `original` or `smote`, so the two phases never overwrite each other.


In [ ]:
# ── Inspect the produced artefacts ────────────────────────────────────────────
print("Output artefacts (per dataset / mode):")
for d in ("results", "figures", "models", "processed_data"):
    p = os.path.join(PROJECT_ROOT, d)
    if not os.path.isdir(p):
        print(f"  {d}/  (missing — run the pipeline cell first)")
        continue
    dsets = sorted(os.listdir(p))
    for ds in dsets:
        dsp = os.path.join(p, ds)
        if not os.path.isdir(dsp):
            print(f"  {d}/{ds}  (file, {os.path.getsize(dsp)} bytes)")
        elif ds == "cross_dataset":
            files = sorted(os.listdir(dsp))
            print(f"  {d}/{ds}/  (files: {', '.join(files) or '—'})")
        else:
            modes = sorted(os.listdir(dsp))
            print(f"  {d}/{ds}/  modes: {', '.join(modes) or '—'}")


## Notes

- **Trial run first (recommended):** set `QUICK_DATASET = "olist"` and Run All —
  the notebook loops just that dataset through both modes so you can confirm your
  datasource + the code before committing to the full sweep. Set it back to
  `None` for the full run.
- **Data:** attach the same inputs as before under `/kaggle/input/datasets`.
  The pre-run validation gate FAILs loudly (and stops Run All) if a required
  input slug is missing.
- **No more zip uploads:** code is pulled live from
  https://github.com/nikhilwankhedee/churn on every run (`UPDATE_REPO = True`).
- **Why rerun these 5?** olist + retailrocket are rerun with the fixed
  user-disjoint temporal split (previous runs overlapped 39.8%/86.4% of test
  users with train); instacart, lastfm and rees46 are the clean datasets
  re-pinned in the rerun scope to reconfirm the untouched pipelines.
- **Ablation study:** credit_card + telco corrected ablations run in
  `run_ablation_study.ipynb` (only their `results/<ds>/<mode>/ablation/*` are
  refreshed — the main models, previously valid, are never rerun).
- **What changed / why:** see the commit + `churn_pipeline_audit_report_2026-08-30.md`.
- **Resume / split runs:** outputs are per dataset and mode, so re-running only
  adds the missing ones (reruns overwrite their own dataset+mode folder in place).
